# Experiment 1: Multimodal Generative AI

CSL75 Skill Enhancement Laboratory.

Theme: a glass bangle furnace at 4 a.m. in Firozabad.

One theme drives three independent generative models. The notebook then scores
how well the image and the audio match the poem, using CLIP and CLAP.

Three conditions are run so the scores can be compared against something:

| Condition | Image and audio prompts built from |
| --- | --- |
| `direct` | the theme string |
| `derived` | a scene description extracted from the generated poem |
| `control` | an unrelated prompt, used as a baseline |

Before running: Runtime, Change runtime type, T4 GPU.

## 1. Check the runtime

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader \
    || echo "No GPU detected. Set Runtime, Change runtime type, T4 GPU."

## 2. Install dependencies

About two minutes on a fresh runtime. If Colab shows a "Restart runtime"
button after this cell, click it, then continue from section 3. Do not re-run
this install cell after restarting.

In [ ]:
!pip install -q "diffusers>=0.30" "transformers>=4.44" "accelerate>=0.33" \
    "librosa>=0.10" "soundfile>=0.12" "cohere>=5.11" "sentencepiece>=0.2"

## 3. Load the experiment modules

Upload `experiment1_multimodal.zip` into `/content` using the folder icon in
the left sidebar, then run this cell. It extracts the archive, puts the modules
on the import path, and reloads any that were already imported.

The archive is the source of truth: extracting overwrites the `.py` files but
leaves the `outputs` directory alone. Run this cell again whenever you replace
the zip or restart the runtime.

In [ ]:
import glob
import importlib
import os
import sys
import zipfile
from pathlib import Path

ARCHIVE = Path("/content/experiment1_multimodal.zip")
MODULES = (
    "config", "prompts", "runtime", "text_stage", "image_stage",
    "audio_stage", "alignment", "report", "run_experiment",
)


def find_project():
    hits = glob.glob("/content/**/run_experiment.py", recursive=True)
    return Path(hits[0]).parent if hits else None


if ARCHIVE.exists():
    with zipfile.ZipFile(ARCHIVE) as archive:
        archive.extractall("/content")

project = find_project()
if project is None:
    raise FileNotFoundError(
        "run_experiment.py was not found under /content. Upload "
        "experiment1_multimodal.zip into /content and run this cell again."
    )

os.chdir(project)
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

reloaded = []
for name in MODULES:
    if name in sys.modules:
        importlib.reload(sys.modules[name])
        reloaded.append(name)

print(f"project dir : {project}")
print(f"modules     : {sorted(f.name for f in project.glob('*.py'))}")
print(f"reloaded    : {reloaded or 'nothing was imported yet'}")

## 4. Language model credentials

A free Cohere trial key produces much better poems than the small local model,
and the poem feeds the derived condition prompts that this experiment is
testing. Get one at dashboard.cohere.com, under API Keys.

The prompt below is masked, so the key is never written into the notebook.
Press Enter on an empty prompt to fall back to a local Hugging Face model.

In [ ]:
import getpass
import os

key = getpass.getpass("Cohere API key (press Enter to skip): ").strip()
if key:
    os.environ["COHERE_API_KEY"] = key
    print("Cohere backend will be used.")
else:
    os.environ.pop("COHERE_API_KEY", None)
    print("No key set. The local Hugging Face model will be used.")

## 5. Run the experiment

Renders three images and three audio clips, then scores them. Roughly eight to
twelve minutes on a T4, longer on the first run because about 6 GB of model
weights are downloaded.

Keep this tab active. Colab disconnects idle free runtimes.

In [ ]:
import run_experiment

THEME = (
    "A glass bangle furnace at 4 a.m. in Firozabad, where workers draw molten "
    "glass from a coal fired furnace and shape it into bangles before sunrise"
)

# Send each backend to its own directory so the runs can be compared later.
OUTPUT = "outputs_cohere"   # use "outputs_local" when you skip the Cohere key

ARGS = [
    "--theme", THEME,
    "--seed", "1729",
    "--audio-seconds", "25",
    "--output", OUTPUT,
]

run_experiment.main(ARGS)

### Recovering from a failure in the scoring stage

If the run got as far as printing "Stage 4 of 4" and then raised, the poem,
images and audio were already written to `outputs`. Fix the problem, re-run
section 3 to reload the modules, then run the cell below. It reuses those
files and runs only the scoring and report stages, which takes about a minute
instead of ten.

In [ ]:
run_experiment.main(ARGS + ["--score-only"])

## 6. Inspect the outputs

In [ ]:
import json
from pathlib import Path

manifest = json.loads(Path(OUTPUT) / "manifest.json".read_text())

print(manifest["poem"])
print()
print(json.dumps(manifest["scene"], indent=2))

In [ ]:
from IPython.display import Audio, Image, display

for condition, path in manifest["images"].items():
    print(f"Image, {condition} condition")
    display(Image(filename=path, width=420))

for condition, path in manifest["audio"].items():
    print(f"Audio, {condition} condition")
    display(Audio(filename=path))

In [ ]:
print(f"{'condition':<10}{'CLIP':>10}{'CLAP':>10}")
for condition in manifest["clip"]:
    print(
        f"{condition:<10}"
        f"{manifest['clip'].get(condition, float('nan')):>10.4f}"
        f"{manifest['clap'].get(condition, float('nan')):>10.4f}"
    )
print()
print("CLIP and CLAP were trained separately. Compare values down a column,")
print("not across. The control row shows what a poor match looks like.")

## 7. Compare two runs

Run the experiment twice with the same theme and seed, once with the Cohere key
and once without, sending each to its own output directory. Then build the
comparison document.

Because the direct and control prompts are constants and the seed is fixed,
those artefacts come out byte identical across runs. Only the derived ones
change, since only they depend on the poem. Any score that moves for an
identical file was moved by the text alone.

In [ ]:
import compare_runs

compare_runs.main([
    "outputs_local",
    "outputs_cohere",
    "--labels", "Qwen", "Cohere",
    "--output", "comparison.md",
])

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path

display(Markdown(Path("comparison.md").read_text()))

## 8. Collect everything for the report

`outputs/report.md` already contains every prompt, output and score. The
rubric table and the discussion sections are left blank on purpose, since
those are the parts that carry the marks.

In [ ]:
import shutil

from google.colab import files

shutil.make_archive(f"experiment1_{OUTPUT}", "zip", OUTPUT)
files.download(f"experiment1_{OUTPUT}.zip")